In [8]:
# Cell 1 — Imports and configuration
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
EMBED_MODEL = "all-MiniLM-L6-v2"
GROQ_MODEL = "openai/gpt-oss-20b"
TOP_K = 3

print("Imports OK")
print("API key loaded:", "YES" if os.getenv("GROQ_API_KEY") else "NO")

Imports OK
API key loaded: YES


In [2]:
# Cell 2 — Load PDF and split into chunks
loader = PyPDFLoader("test.pdf")
pages = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = splitter.split_documents(pages)

print(f"Loaded {len(pages)} pages")
print(f"Created {len(chunks)} chunks")
print(f"\nFirst chunk preview:\n{chunks[0].page_content[:300]}...")

Loaded 11 pages
Created 118 chunks

First chunk preview:
MNRAS000, 1–11 (2026) Preprint 2 October 2026 Compiled using MNRAS L ATEX style file v3.3
Interpreting effective homogeneous rheologies through a local differential
map with application to the Moon
Yeva Gevorgyan1★
1King Abdullah University of Science and Technology (KAUST), Thuwal 23955-6900, Saudi...


In [3]:
# Cell 3 — Build the vector store
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"}
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print(f"Vector store ready: {vectorstore._collection.count()} vectors")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\mkaru\Dropbox\Projects\documind\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mkaru\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store ready: 118 vectors


In [9]:
# Cell 4 — Build the RAG chain
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K}
)

llm = ChatGroq(
    model_name=GROQ_MODEL,
    temperature=0,
    groq_api_key=os.getenv("GROQ_API_KEY")
)

template = """Answer the question based ONLY on the following context.
If the answer is not in the context, say "I don't have enough information to answer that."

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready")

RAG chain ready


In [10]:
# Cell 5 — Ask a question
question = "What is the main topic of this document?"
answer = rag_chain.invoke(question)
print(answer)

The document focuses on the effective rheology of the Moon’s interior, specifically a five‑layer Maxwell viscoelastic model and the analysis of its relaxation times and singular values.


In [11]:
print(rag_chain.invoke("What is the capital of France?"))


I don't have enough information to answer that.


In [12]:
print(rag_chain.invoke("What are the five layers of the model?"))

The five layers of the model are:

1. Solid inner core  
2. Liquid outer core  
3. Low‑viscosity zone  
4. Upper mantle  
5. Crust


In [13]:
print(rag_chain.invoke("Summarize this document in 3 bullet points."))

- The study presents a five‑layer lunar Maxwell model (solid inner core, liquid outer core, low‑viscosity zone, upper mantle, crust) with radii, densities, rigidities, and viscosities taken from Matsuyama et al. (2016) and Gevorgyan et al. (2023; 2026c).  
- The model focuses on the mantle’s viscoelastic properties, varying its viscosity (ηₘ) and rigidity (μₘ) to compute the complex shear modulus \(b_{\mu_m}(s)\) and the characteristic equation \(A(v)s^2+B(v)s+C(v)=0\).  
- From this characteristic equation the Love number \(k_2(s)\) is derived, linking the mantle’s rheology to observable tidal responses.
